# 05 — RAG completo con ChromaDB

> **Bloque:** LLMs | **Nivel:** Intermedio-Avanzado
>
> Complementario al tutorial [05-rag-chromadb.md](../../llms/05-rag-chromadb.md)

Pipeline RAG completo: indexamos documentos, hacemos búsqueda semántica y generamos respuestas fundamentadas.

In [ ]:
# %pip install chromadb anthropic sentence-transformers python-dotenv

import chromadb
import anthropic
import json
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

client_ai = anthropic.Anthropic()
client_db = chromadb.EphemeralClient()  # En memoria (sin persistencia) para el notebook

print(f'✓ ChromaDB {chromadb.__version__} | Anthropic SDK listo')

## 1. Base de conocimiento de ejemplo

In [ ]:
DOCUMENTOS = [
    {
        'fuente': 'politica_vacaciones',
        'texto': 'Los empleados tienen derecho a 23 días laborables de vacaciones anuales. '
                 'Las vacaciones se solicitan con 15 días de antelación a través del portal de RRHH. '
                 'En agosto se garantizan al menos 10 días consecutivos. '
                 'Los días no disfrutados pueden trasladarse al primer trimestre del año siguiente.'
    },
    {
        'fuente': 'sobre_empresa',
        'texto': 'TechCorp fue fundada en 2010 en Madrid. Cuenta con 150 empleados. '
                 'Opera en 8 países europeos y tiene oficinas en Madrid, Barcelona y Lisboa. '
                 'Su producto principal es TechCloud, una plataforma SaaS de gestión empresarial. '
                 'La empresa facturó 12 millones de euros en 2025.'
    },
    {
        'fuente': 'politica_teletrabajo',
        'texto': 'El modelo de trabajo es híbrido: 3 días en oficina y 2 días en remoto por semana. '
                 'Los martes y jueves son días de presencia obligatoria para todos los equipos. '
                 'El teletrabajo desde el extranjero está permitido hasta 30 días al año '
                 'previa aprobación del responsable directo.'
    },
    {
        'fuente': 'beneficios_empleados',
        'texto': 'TechCorp ofrece seguro médico privado desde el primer día. '
                 'Ticket restaurante de 11€ por día trabajado. '
                 'Presupuesto anual de 1.500€ para formación y certificaciones. '
                 'Gimnasio gratuito en las oficinas de Madrid y Barcelona.'
    },
    {
        'fuente': 'proceso_bajas',
        'texto': 'Las bajas por enfermedad deben comunicarse al responsable antes de las 9:00. '
                 'El parte médico debe entregarse en RRHH en los 3 días siguientes. '
                 'A partir del cuarto día de baja, la Seguridad Social asume el 60% del salario. '
                 'La empresa complementa hasta el 100% durante los primeros 30 días.'
    }
]

print(f'Documentos disponibles: {len(DOCUMENTOS)}')
for d in DOCUMENTOS:
    print(f'  - {d["fuente"]}: {len(d["texto"].split())} palabras')

## 2. Indexar documentos en ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer

print('Cargando modelo de embeddings...')
modelo_emb = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print(f'✓ Modelo cargado (dimensión: {modelo_emb.get_sentence_embedding_dimension()})')

# Crear colección
coleccion = client_db.create_collection('base_conocimiento', metadata={'hnsw:space': 'cosine'})

# Indexar
textos  = [d['texto'] for d in DOCUMENTOS]
fuentes = [d['fuente'] for d in DOCUMENTOS]
ids     = [f'doc_{i}' for i in range(len(DOCUMENTOS))]

embeddings = modelo_emb.encode(textos).tolist()

coleccion.add(
    ids=ids,
    embeddings=embeddings,
    documents=textos,
    metadatas=[{'fuente': f} for f in fuentes]
)

print(f'✓ {coleccion.count()} documentos indexados')

## 3. Búsqueda semántica

In [ ]:
def buscar(pregunta: str, top_k: int = 3) -> list:
    emb = modelo_emb.encode([pregunta]).tolist()
    res = coleccion.query(query_embeddings=emb, n_results=top_k,
                          include=['documents', 'metadatas', 'distances'])
    return [
        {'texto': doc, 'fuente': meta['fuente'], 'similitud': round(1 - dist, 4)}
        for doc, meta, dist in zip(
            res['documents'][0], res['metadatas'][0], res['distances'][0]
        )
    ]

# Probar búsqueda
preguntas_test = [
    '¿Cuántos días de vacaciones tengo al año?',
    '¿Puedo trabajar desde casa todos los días?',
    '¿Qué pasa si me pongo enfermo?',
]

for p in preguntas_test:
    resultados = buscar(p, top_k=2)
    print(f'\n🔍 "{p}"')
    for r in resultados:
        print(f'  [{r["similitud"]:.1%}] {r["fuente"]}: {r["texto"][:80]}...')

## 4. Generación de respuesta fundamentada

In [ ]:
def rag(pregunta: str, top_k: int = 3, umbral: float = 0.3) -> dict:
    # Recuperar fragmentos
    fragmentos = [f for f in buscar(pregunta, top_k) if f['similitud'] >= umbral]

    if not fragmentos:
        return {'respuesta': 'No tengo información sobre esto en la base de conocimiento.',
                'fuentes': []}

    contexto = '\n\n'.join([
        f'[{f["fuente"]}]\n{f["texto"]}' for f in fragmentos
    ])

    prompt = f"""Responde la pregunta basándote ÚNICAMENTE en el contexto.
Si la respuesta no está en el contexto, di "No tengo esa información".

<contexto>
{contexto}
</contexto>

Pregunta: {pregunta}"""

    r = client_ai.messages.create(
        model='claude-haiku-4-5-20251001', max_tokens=400, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return {
        'respuesta': r.content[0].text,
        'fuentes': list({f['fuente'] for f in fragmentos}),
        'num_fragmentos': len(fragmentos)
    }

# Demostración
preguntas = [
    '¿Cuántos días de vacaciones tengo?',
    '¿Puedo teletrabajar desde otro país?',
    '¿Cuál es el salario mínimo de la empresa?',  # No está en los documentos
]

for p in preguntas:
    print(f'\n❓ {p}')
    res = rag(p)
    print(f'💬 {res["respuesta"]}')
    print(f'📚 Fuentes: {res["fuentes"]}')

## 5. Visualizar similitudes

In [ ]:
preguntas_viz = [
    '¿Cuántos días de vacaciones?',
    '¿Seguro médico?',
    '¿Teletrabajo desde fuera?',
    '¿Baja por enfermedad?',
]

fig, axes = plt.subplots(1, len(preguntas_viz), figsize=(16, 4))

for ax, pregunta in zip(axes, preguntas_viz):
    resultados = buscar(pregunta, top_k=len(DOCUMENTOS))
    fuentes = [r['fuente'].replace('_', '\n') for r in resultados]
    scores  = [r['similitud'] for r in resultados]
    colores = ['#2ECC71' if s >= 0.5 else '#F39C12' if s >= 0.3 else '#E74C3C' for s in scores]

    ax.barh(fuentes, scores, color=colores, alpha=0.85)
    ax.set_xlim(0, 1)
    ax.set_title(pregunta, fontsize=9)
    ax.axvline(0.3, color='red', linestyle='--', alpha=0.5, linewidth=0.8)

plt.suptitle('Similitud semántica por pregunta', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Verde ≥ 0.5 (alta), Naranja ≥ 0.3 (media), Rojo < 0.3 (baja, no se usa)')

## 6. Añade tus propios documentos

In [ ]:
# Añade aquí tu propio documento
mi_documento = """
Escribe aquí el contenido de tu documento...
"""
mi_fuente = 'mi_documento'

if len(mi_documento.split()) > 5:  # Solo si no es el placeholder
    emb = modelo_emb.encode([mi_documento]).tolist()
    coleccion.add(
        ids=[f'custom_{coleccion.count()}'],
        embeddings=emb,
        documents=[mi_documento],
        metadatas=[{'fuente': mi_fuente}]
    )
    print(f'✓ Documento añadido. Total: {coleccion.count()}')

    # Prueba una pregunta sobre tu documento
    mi_pregunta = '¿Qué dice el documento sobre ...?'
    print(rag(mi_pregunta)['respuesta'])
else:
    print('Modifica mi_documento con tu propio contenido para probarlo.')

---
**Tutorial relacionado:** [05-rag-chromadb.md](../../llms/05-rag-chromadb.md)